# Layer 3: reliability analysis

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

for _cand in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_cand / "layer3_reliability.py").exists():
        sys.path.insert(0, str(_cand))
        break
import layer3_reliability as rel

ROOT = next((p for p in [Path("/workspace"), Path.cwd(), Path.cwd().parent]
             if (p / "output" / "14_layer2_crossfarm_eval").exists()), Path.cwd())

In [ ]:
PRED_PATH = ROOT / "output" / "14_layer2_crossfarm_eval" / "crossfarm_fold_predictions.csv.gz"
TOKENS_PATH = ROOT / "output" / "05_tokens_refined" / "token_counts_by_clip.csv"
OUT_CALIB = ROOT / "output" / "19_layer3_reliability_analysis"
OUT_SELECT = ROOT / "output" / "20_layer3_selective_prediction"
OUT_DICT = ROOT / "output" / "21_layer3_dictionary"
OUT_SUMMARY = ROOT / "output" / "22_layer3_confidence_review"
for d in (OUT_CALIB, OUT_SELECT, OUT_DICT, OUT_SUMMARY):
    d.mkdir(parents=True, exist_ok=True)

pred = pd.read_csv(PRED_PATH)
token_counts = pd.read_csv(TOKENS_PATH)
TARGETS = ["is_nonvocal", "is_distress_related", "is_resource_related", "proxy_functional_v1"]
COVERAGES = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def composite_pick(rows, keys=("ece", "brier", "logloss")):
    ranked = {k: sorted(range(len(rows)), key=lambda i: rows[i][k]) for k in keys}
    score = [0] * len(rows)
    for k in keys:
        for r, idx in enumerate(ranked[k]):
            score[idx] += r
    return int(np.argmin(score))

In [ ]:
def target_frame(target):
    sub = pred[pred.target_name == target].reset_index(drop=True)
    task = sub.task_type.iloc[0]
    farms = sub.heldout_farm.to_numpy()
    clip_ids = sub["_record_key"].to_numpy()
    if task == "binary":
        y = sub.y_true.to_numpy().astype(int)
        p1 = sub["proba__1"].to_numpy()
        P = np.c_[1 - p1, p1]
        class_names = ["0", "1"]
    else:
        class_cols = sorted(c for c in sub.columns if c.startswith("proba__") and c != "proba__1")
        class_names = [c.replace("proba__", "") for c in class_cols]
        P = sub[class_cols].to_numpy()
        y = sub.y_true_label.map({n: i for i, n in enumerate(class_names)}).to_numpy().astype(int)
    return {"task": task, "farms": farms, "clip_ids": clip_ids, "y": y, "P": P, "class_names": class_names}

In [ ]:
calibration_rows = []
calibrated_store = {}

for target in TARGETS:
    tf = target_frame(target)
    y, farms, P = tf["y"], tf["farms"], tf["P"]
    n_classes = len(tf["class_names"])
    methods = rel.BINARY_METHODS if tf["task"] == "binary" else rel.MULTICLASS_METHODS

    candidates = []
    for method in methods:
        if tf["task"] == "binary":
            cp1 = rel.group_aware_binary(farms, P[:, 1], y, method)
            CP = np.c_[1 - cp1, cp1]
        else:
            CP = rel.group_aware_multiclass(farms, P, y, method, n_classes)
        pred_lbl = CP.argmax(1)
        conf = CP.max(1)
        candidates.append({
            "method": method,
            "ece": rel.expected_calibration_error(y, conf, pred_lbl),
            "brier": rel.brier_binary(y, CP[:, 1]) if tf["task"] == "binary"
                     else rel.brier_multiclass(y, CP, n_classes),
            "logloss": rel.log_loss_safe(y, CP, n_classes),
            "CP": CP,
        })
    raw = candidates[0]
    best = candidates[composite_pick(candidates)]
    calibrated_store[target] = {"CP": best["CP"], "method": best["method"], "raw_P": P}
    calibration_rows.append({
        "target": target, "selected_method": best["method"],
        "ece_raw": raw["ece"], "ece_selected": best["ece"],
        "brier_raw": raw["brier"], "brier_selected": best["brier"],
        "logloss_raw": raw["logloss"], "logloss_selected": best["logloss"],
    })

calibration_df = pd.DataFrame(calibration_rows)

In [ ]:
selective_rows = []
coverage_rows = []

for target in TARGETS:
    tf = target_frame(target)
    y = tf["y"]
    raw_P = calibrated_store[target]["raw_P"]
    cal_P = calibrated_store[target]["CP"]
    options = {
        "raw": (raw_P.argmax(1), raw_P.max(1)),
        "calibrated": (cal_P.argmax(1), cal_P.max(1)),
    }
    scored = {}
    for tag, (yp, conf) in options.items():
        curve = rel.selective_prediction(y, yp, conf, COVERAGES)
        for row in curve:
            coverage_rows.append({"target": target, "score": tag, **row})
        base = f1_score(y, yp, average="macro", zero_division=0)
        at80 = next(r for r in curve if r["coverage_target"] == 0.80)
        scored[tag] = (base, at80)
    best_tag = max(scored, key=lambda t: scored[t][1]["retained_macro_f1"])
    base, at80 = scored[best_tag]
    selective_rows.append({
        "target": target, "score_version": best_tag,
        "baseline_macro_f1": base,
        "retained_macro_f1_80cov": at80["retained_macro_f1"],
        "selective_gain": at80["retained_macro_f1"] - base,
        "abstention_rate": at80["abstention_rate"],
    })

selective_df = pd.DataFrame(selective_rows)
coverage_df = pd.DataFrame(coverage_rows)

In [ ]:
unit_membership = token_counts[["clip_id", "unit_id"]].drop_duplicates()

dictionary_rows = []
tier_distribution = []

for target in TARGETS:
    tf = target_frame(target)
    sel = selective_df[selective_df.target == target].iloc[0]
    cal = calibrated_store[target]
    P = cal["raw_P"] if sel.score_version == "raw" else cal["CP"]
    conf = P.max(1)
    clip_conf = pd.DataFrame({"clip_id": tf["clip_ids"], "confidence": conf})
    clip_conf["uncertain"] = rel.deferred_mask(conf, 0.80).astype(int)

    merged = unit_membership.merge(clip_conf, on="clip_id", how="inner")

    counts = {"HIGH": 0, "MEDIUM": 0, "LOW": 0, "UNCERTAIN": 0}
    for unit_id, grp in merged.groupby("unit_id"):
        mean_conf = grp["confidence"].mean()
        unc_rate = grp["uncertain"].mean()
        n_clips = len(grp)
        if unc_rate <= 0.15:
            tier = "HIGH"
        elif unc_rate <= 0.35:
            tier = "MEDIUM"
        elif unc_rate <= 0.50:
            tier = "LOW"
        else:
            tier = "UNCERTAIN"
        counts[tier] += 1
        dictionary_rows.append({"target": target, "unit_id": unit_id, "n_clips": n_clips,
                                "mean_confidence": mean_conf, "uncertain_rate": unc_rate, "tier": tier})
    total = sum(counts.values())
    tier_distribution.append({"target": target, "n_units": total,
                              **{f"{k}_pct": round(100 * v / total, 1) for k, v in counts.items()}})

dictionary_df = pd.DataFrame(dictionary_rows)
tier_distribution_df = pd.DataFrame(tier_distribution)

In [ ]:
master_rows = []
for target in TARGETS:
    c = calibration_df[calibration_df.target == target].iloc[0]
    s = selective_df[selective_df.target == target].iloc[0]
    lofo = pred[(pred.target_name == target)]
    if lofo.task_type.iloc[0] == "binary":
        lofo_macro = f1_score(lofo.y_true, lofo.y_pred, average="macro", zero_division=0)
    else:
        lofo_macro = f1_score(lofo.y_true_label, lofo.y_pred_label, average="macro", zero_division=0)
    master_rows.append({
        "target": target,
        "lofo_macro_f1": round(lofo_macro, 3),
        "calibration_method": c.selected_method,
        "ece_raw": round(c.ece_raw, 3), "ece_selected": round(c.ece_selected, 3),
        "selective_score": s.score_version,
        "retained_macro_f1_80cov": round(s.retained_macro_f1_80cov, 3),
        "selective_gain": round(s.selective_gain, 3),
    })
master_df = pd.DataFrame(master_rows)

In [ ]:
calibration_df.to_csv(OUT_CALIB / "calibration_summary.csv", index=False)
selective_df.to_csv(OUT_SELECT / "selective_prediction_summary.csv", index=False)
coverage_df.to_csv(OUT_SELECT / "selective_coverage_curves.csv", index=False)
dictionary_df.to_csv(OUT_DICT / "dictionary_unit_tiers.csv", index=False)
tier_distribution_df.to_csv(OUT_DICT / "dictionary_tier_distribution.csv", index=False)
master_df.to_csv(OUT_SUMMARY / "layer3_reliability_master_summary.csv", index=False)

manifest = {"build_ts_utc": RUN_TS, "targets": TARGETS, "coverages": COVERAGES,
            "prediction_source": str(PRED_PATH.name)}
with open(OUT_SUMMARY / "layer3_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

In [ ]:
MASTER_PATH = ROOT / "output" / "09_layer2_dataset" / "layer2_master_table.csv.gz"
master_idx = pd.read_csv(MASTER_PATH).set_index("_record_key")

def _norm_entropy(P):
    Pc = np.clip(P, 1e-12, 1.0)
    return (-np.sum(Pc * np.log(Pc), axis=1)) / np.log(P.shape[1])

confidence_rows, perfarm_rows, reason_rows, polysemous_rows = [], [], [], []
sel_by_target = selective_df.set_index("target")["score_version"].to_dict()

for target in TARGETS:
    tf = target_frame(target)
    y, farms, clip_ids = tf["y"], tf["farms"], tf["clip_ids"]
    raw_P = calibrated_store[target]["raw_P"]
    cal_P = calibrated_store[target]["CP"]
    scoreP = raw_P if sel_by_target[target] == "raw" else cal_P
    yp = scoreP.argmax(1)
    raw_pred = raw_P.argmax(1)
    correct = (yp == y).astype(int)
    raw_conf, cal_conf, sel_conf = raw_P.max(1), cal_P.max(1), scoreP.max(1)

    confidence_rows.append({
        "target": target,
        "mean_raw_confidence": float(raw_conf.mean()),
        "mean_calibrated_confidence": float(cal_conf.mean()),
        "mean_confidence_correct": float(sel_conf[correct == 1].mean()),
        "mean_confidence_incorrect": float(sel_conf[correct == 0].mean()),
        "accuracy": float(correct.mean()),
    })

    for f in sorted(set(farms.tolist())):
        m = farms == f
        perfarm_rows.append({"target": target, "heldout_farm": f,
                             "n_clips": int(m.sum()),
                             "accuracy": float((raw_pred[m] == y[m]).mean())})

    ent = _norm_entropy(scoreP)
    ms = master_idx.reindex(clip_ids)
    dominance = (ms["top_token_count"] / ms["n_segments"]).to_numpy()
    diversity = ms["n_unique_tokens"].to_numpy()
    flags = {
        "high_predictive_entropy": ent > np.quantile(ent, 0.75),
        "low_confidence": rel.deferred_mask(sel_conf, 0.80),
        "repeated_token_dominance": dominance > 0.5,
        "low_token_diversity": diversity <= 3,
    }
    for reason, mask in flags.items():
        reason_rows.append({"target": target, "reason": reason,
                            "n_clips": int(np.nansum(np.asarray(mask, dtype=float)))})

    clip_df = pd.DataFrame({"clip_id": clip_ids, "pred": yp, "entropy": ent})
    mu = unit_membership.merge(clip_df, on="clip_id", how="inner")
    stats = mu.groupby("unit_id").agg(
        mean_entropy=("entropy", "mean"),
        dominant_share=("pred", lambda s: s.value_counts(normalize=True).max()),
        n_clips=("pred", "size")).reset_index()
    ent_cut = stats["mean_entropy"].quantile(0.85)
    poly = stats[(stats["mean_entropy"] >= ent_cut) & (stats["dominant_share"] < 0.60)]
    polysemous_rows.append({"target": target, "n_polysemous": int(len(poly)),
                            "polysemous_units": ";".join(poly["unit_id"].astype(str).tolist())})

confidence_summary_df = pd.DataFrame(confidence_rows)
perfarm_accuracy_df = pd.DataFrame(perfarm_rows)
uncertainty_reasons_df = pd.DataFrame(reason_rows)
polysemous_df = pd.DataFrame(polysemous_rows)

confidence_summary_df.to_csv(OUT_SUMMARY / "confidence_summary.csv", index=False)
perfarm_accuracy_df.to_csv(OUT_SUMMARY / "per_farm_accuracy.csv", index=False)
uncertainty_reasons_df.to_csv(OUT_SUMMARY / "uncertainty_reasons_summary.csv", index=False)
polysemous_df.to_csv(OUT_DICT / "polysemous_candidates.csv", index=False)

print("Confidence review outputs written.")
print(confidence_summary_df.round(3).to_string(index=False))
print(uncertainty_reasons_df.pivot(index="reason", columns="target", values="n_clips"))
print(polysemous_df.to_string(index=False))

In [ ]:
print("CALIBRATION")
display(calibration_df.round(3))
print("SELECTIVE PREDICTION @80% coverage")
display(selective_df.round(3))
print("DICTIONARY TIER DISTRIBUTION")
display(tier_distribution_df)
print("MASTER RELIABILITY SUMMARY")
display(master_df)